# Metabolite Hit Validation (external cohorts)
Here I will read in metacardis data with available antibiotic use information and see how my findings compare. 

Sarah has also performed this analysis for the SOL cohort. She's send me her findings, and I will combine all data here for plotting in R.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
# Now you can import your file
from regression_functions import *

# Metacardis

## Load the excel sheets into pandas DFs and save as .csv

In [2]:
'''These data come from Fromentin 2022'''
##############################################
# microbes = pd.read_excel("41591_2022_1688_MOESM3_ESM.xlsx", sheet_name=10, header=1)
# metabolites = pd.read_excel("41591_2022_1688_MOESM3_ESM.xlsx", sheet_name=12, header=1)
# clients = pd.read_excel("41591_2022_1688_MOESM3_ESM.xlsx", sheet_name=9, header=1)
#other_drugs = pd.read_excel("metacardis_data/41591_2022_1688_MOESM3_ESM.xlsx", sheet_name=14, header=1)

'These data come from Fromentin 2022'

In [3]:
# microbes.to_csv("metacardis_microbes.csv", index=False)
# metabolites.to_csv("metacardis_metabolites.csv", index=False)
# clients.to_csv("metacardis_client_metadata.csv", index=False)
#other_drugs.to_csv("metacardis_data/fromentin_metacardis_other_drugs.csv", index=False)

In [4]:
'''These data come from Forslund 2021'''
##############################################
# abx_count = pd.read_excel("metacardis_data/Table1-4_Metadata.xlsx", sheet_name=6, header=0)
# group_char = pd.read_excel("metacardis_data/Table1-4_Metadata.xlsx", sheet_name=1, header=0)
# individ_metadata = pd.read_excel("metacardis_data/Table1-4_Metadata.xlsx", sheet_name=2, header=0)
# anti_diab = pd.read_excel("metacardis_data/Table1-4_Metadata.xlsx", sheet_name=4, header=0)

'These data come from Forslund 2021'

In [5]:
# abx_count.to_csv("metacardis_data/metacardis_abx_count.csv", index=False)
# group_char.to_csv("metacardis_data/metacardis_group_characteristics.csv", index=False)
# individ_metadata.to_csv("metacardis_data/metacardis_2021_client_metadata.csv", index=False)
# anti_diab.to_csv("metacardis_data/forslund_metacardis_other_med_use.csv", index=False)

In [6]:
# Have not implemented this yet
# individ_metadata_small = [['ID', 'PatientGroup', 'BMI', 'AGE', 'CENTER', 'pa_work_2cl', 'aHEI', 'Gender']]

## Check/Clean the DFs

In [7]:
clients_fromentin2022 = pd.read_csv("metacardis_data/metacardis_client_metadata.csv")
microbes = pd.read_csv("metacardis_data/metacardis_microbes.csv")
metabolites = pd.read_csv("metacardis_data/metacardis_metabolites.csv")

In [8]:
clients_forslund2021 = pd.read_csv("metacardis_data/metacardis_2021_client_metadata.csv")
abx_count = pd.read_csv("metacardis_data/metacardis_abx_count.csv")
group_char = pd.read_csv("metacardis_data/metacardis_group_characteristics.csv")
anti_diab = pd.read_csv("metacardis_data/forslund_metacardis_other_med_use.csv")

## Merge client metadata from both studies

### First start with Fromentin 2022

In [9]:
clients_fromentin2022['Status'] = clients_fromentin2022['Status'].str.replace(r'\d+', '', regex=True)

In [10]:
clients_fromentin2022 = clients_fromentin2022.rename(columns = {
    'Age (years)':'AGE',
    'BMI (kg/m²)':'BMI',
    'Nationality':'CENTER',
    'Physical activity (h/week)':'activity_h_week',
    'Diabetic status':'diabetic_status'
})

In [11]:
len(clients_fromentin2022[clients_fromentin2022['Gender'].isna()]) # All IHD or ACS Patients anyways

42

In [12]:
len(clients_fromentin2022)

1882

In [13]:
clients_fromentin2022 = clients_fromentin2022.dropna(subset='Gender')

In [14]:
len(clients_fromentin2022)

1840

In [15]:
# Subset col of interest from Fromentin 2022
clients_fromentin_clean = (clients_fromentin2022[[
    'ID','Status','AGE', 'BMI','Gender', 'CENTER','activity_h_week','diabetic_status'
]].copy())


clients_fromentin_clean.loc[clients_fromentin_clean.CENTER == 'Danemark', 'CENTER'] = 'Denmark'

### Moving onto Forslund 2021

In [16]:
len(clients_forslund2021)

2173

In [17]:
clients_forslund2021 = clients_forslund2021.dropna(subset='Gender')

In [18]:
len(clients_forslund2021)

1614

In [19]:
# This cell determines how many IDs are shared between the two studies
clients_forslund2021.rename(columns={'SampleID': 'ID'}, inplace=True)
test = clients_forslund2021[['ID','Gender']].merge(clients_fromentin_clean[['ID', 'Gender']], on=['ID'], how='inner').drop_duplicates()
len(test)
# Also one can see here that 1.0 maps to male and 0.0 maps to female

611

Clean up data so that values from the two papers match (like gender)

In [20]:
# Change gender encoding in Forslund 2021 paper metadata 
clients_forslund2021['Gender_text'] = clients_forslund2021['Gender'].apply(
    lambda x: 'Male' if x == 1 else 
    'Female' if x == 0
    else x)
clients_forslund2021.drop(columns='Gender',inplace=True)
clients_forslund2021.rename(columns={'Gender_text': 'Gender'}, inplace=True)

In [21]:
# Outer merge to keep all folks
# Not merging on multiple columns bc they are filled to varying extents
all_clients = (clients_forslund2021.merge(
    clients_fromentin_clean, 
    how='outer', 
    on=['ID'], 
    suffixes=('_df1', '_df2')).drop_duplicates())

# Combine complementary columns
# Overrides NaNs for col that have data
shared_col = ['CENTER', 'AGE', 'BMI', 'Gender']

for col in shared_col: 
    all_clients[col] = all_clients[f'{col}_df1'].combine_first(all_clients[f'{col}_df2'])

# Drop the temp col
(all_clients.drop(columns=
                  [col + '_df1' for col in shared_col] + 
                  [col + '_df2' for col in shared_col], 
                  inplace=True))

all_clients.sort_values(by='ID')

,ID,PatientGroup,SMOKE,pa_work_2cl,aHEI,DASH,DDS,Status,activity_h_week,diabetic_status,CENTER,AGE,BMI,Gender
0,x10MCx1114,3,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,France,59.0,38.154392,Female
1,x10MCx1132,3,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,France,73.0,44.634475,Female
2,x10MCx1133,3,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,France,73.0,NaN,Male
3,x10MCx1134,2a,0.0,1.0,18.144745,36.041306,12.0,NaN,NaN,NaN,France,42.0,44.365888,Female
4,x10MCx1135,3,0.0,1.0,NaN,NaN,NaN,MMC,0.0,Yes,France,57.0,27.719670,Male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2839,x30MCx3388,NaN,NaN,NaN,NaN,NaN,NaN,HF,0.0,Yes,Denmark,72.0,32.463770,Male
2838,x30MCx3388,NaN,NaN,NaN,NaN,NaN,NaN,IHD,0.0,Yes,Denmark,72.0,32.463770,Male
2840,x30MCx3414,NaN,NaN,NaN,NaN,NaN,NaN,HC,18.0,No,Denmark,43.0,25.022160,Male
2841,x30MCx3415,NaN,NaN,NaN,NaN,NaN,NaN,IHD,NaN,No,Denmark,51.0,27.623460,Male


### Let's remove the heart disease and ambiguous phenotypes

In [22]:
patient_group_dict = {
    '1':'metabolic_syndrome', # MetS without T2D or CAD
    '2a':'severe_obesity',    # without T2D or CAD
    '2b':'morbid_obesity',    # Eligible for bariatric surgery but mixed T2D
    '3':'t2dm',               # Without CAD
    '4':'acute_cad',          # With a first acute CAD event
    '5':'chronic_cad',        # Chronic CAD w/ EF >= 45
    '6':'hf',                 # HF w/ EF < 45
    '7':'non_cad_HF',         # Non-CAD related HF
    '8':'control'             # Controls no CAD, T2D, MetS, and BMI <25
}

In [23]:
# Drop heart patients
HC_MMC_UMMC = all_clients[~(all_clients['Status'].isin(
    ['IHD','CIHD','HF', 'ACS']))]

HC_MMC_UMMC_filtered = HC_MMC_UMMC[~(HC_MMC_UMMC['PatientGroup'].isin(
    ['2b','4','5','6','7']))].drop_duplicates()

In [24]:
HC_MMC_UMMC_filtered.PatientGroup.value_counts()

PatientGroup
3     601
2a    291
8     242
1     124
Name: count, dtype: int64

Patient group 3 are all individuals with T2DM, so we should see a 1:1 relationship with the diabetic status annotation

In [25]:
pd.crosstab(HC_MMC_UMMC_filtered['PatientGroup'], HC_MMC_UMMC_filtered['diabetic_status']) # Awesome

diabetic_status,No,Yes
PatientGroup,,
1,59,0
2a,71,0
3,0,130
8,225,0


When running our regressions, we may want to consider which covariate (status or health group) to use. It seems like more individuals have a health group annotation compared to the status.

In [26]:
# HC_MMC_UMMC_filtered.sort_values(by='ID') # Some people have conflicting status annotations!

In [27]:
HC_MMC_UMMC_filtered.Status.isna().sum()

773

In [28]:
HC_MMC_UMMC_filtered.PatientGroup.isna().sum()

357

And generally, it looks like people respond to at least one of those columns:

In [29]:
HC_MMC_UMMC_filtered[(HC_MMC_UMMC_filtered['PatientGroup'].isna()) & (HC_MMC_UMMC_filtered['Status'].isna())]

,ID,PatientGroup,SMOKE,pa_work_2cl,aHEI,DASH,DDS,Status,activity_h_week,diabetic_status,CENTER,AGE,BMI,Gender


## Identify missing health group
Because the health group phenotypes are much more specific, we may be able to annotate some of them on our own using the criterion listed in Forslund et al. 

In [30]:
patient_group_impute = list(set(HC_MMC_UMMC_filtered[(HC_MMC_UMMC_filtered['PatientGroup'].isna())].ID.tolist()))

In [31]:
fromentin_health_data = clients_fromentin2022[['ID','diabetic_status','BMI', 'Fasting plasma triglycerides (mmol/L)', 'HbA1c (%)']].copy()
unknown_group_data = fromentin_health_data[fromentin_health_data.ID.isin(patient_group_impute)].drop_duplicates()

In [32]:
# unknown_group_data[(unknown_group_data.diabetic_status == 'No') & (unknown_group_data.BMI < 25)]

#### Add Waldron Lab annotations
Since there are no LDL annotations on the CSV from Fromentin, let's add the annotation from Waldon lab.

In [33]:
mc_data_waldron = pd.read_csv('../2024_cperez_antibiotics/abx_recency_regressions/data/MetaCardis_2020_a_sample.tsv', sep='\t')
mc_data_waldron['subject_id'] = mc_data_waldron['subject_id'].str[2:]

In [34]:
waldron_filtered = mc_data_waldron[[
    'subject_id','control', 'disease',
    'antibiotics_current_use', 'country',
    'biomarker_name', 'biomarker_unit', 'biomarker_value',
]].copy()

In [35]:
waldron_filtered.dropna(subset='biomarker_name', inplace=True)

# Split the lab values to be apart
waldron_filtered['biomarker_name'] = waldron_filtered['biomarker_name'].str.split(";")
waldron_filtered['biomarker_unit'] = waldron_filtered['biomarker_unit'].str.split(";")
waldron_filtered['biomarker_value'] = waldron_filtered['biomarker_value'].str.split(";")

# Each have their own column
waldron_exploded = waldron_filtered.explode(['biomarker_name', 'biomarker_unit', 'biomarker_value']).copy()

In [36]:
waldron_exploded.biomarker_value = waldron_exploded.biomarker_value.astype(float)
waldron_exploded = waldron_exploded.set_index("subject_id")

# Reshape the data from long format to wide format
waldron_pivoted = waldron_exploded.pivot(
    columns='biomarker_name', 
    values=['biomarker_unit', 'biomarker_value']
)

waldron_pivoted.columns = [f"{biomarker}_{metric}" for biomarker, metric in waldron_pivoted.columns]

In [37]:
waldron_pivoted = waldron_pivoted.reset_index()

In [38]:
waldron_pivoted.rename(columns={
    'biomarker_value_LDL_Particles':'LDL_mg_dl',
    'biomarker_value_Triglyceride':'triglyceride_mg_dl',
    'biomarker_value_Glycosylated_Hemoglobin':'hba1c'
}, inplace=True)

In [39]:
waldron_exploded.drop(columns=['biomarker_name','biomarker_unit','biomarker_value'], inplace=True)

In [40]:
waldron_annot_final = waldron_exploded.merge(
    waldron_pivoted[['subject_id','hba1c', 'LDL_mg_dl','triglyceride_mg_dl']].copy(),
    how='left',
    on='subject_id'
).drop_duplicates()

waldron_annot_final.rename(columns={'subject_id':'ID'}, inplace=True)

Great! Let's grab the users for which the Patient Group is not known and inspect their labs more closely.

In [41]:
unknown_group_data_waldron = unknown_group_data.merge(waldron_annot_final, how='left', on='ID')

In [42]:
# Dropping these as we don't have specific guidelines in Forslund et al. for how to classify these
unknown_group_data_waldron = unknown_group_data_waldron[unknown_group_data_waldron.disease != 'Glucose Intolerance']

In [43]:
# Group 8 Annotations
# Find the IDs
group8 = (unknown_group_data_waldron[unknown_group_data_waldron['disease'] == 'Healthy'].ID)
# Annotate them
HC_MMC_UMMC_filtered.loc[(HC_MMC_UMMC_filtered.ID.isin(group8)), 'PatientGroup'] = '8'
# Remove from pool
unknown_group_data_waldron = unknown_group_data_waldron[unknown_group_data_waldron.disease != 'Healthy']

In [44]:
# Move onto groups 1 and 2a
# None of these folks have T2DM
# Split based on the BMI cut-off listed in the paper/STable 1a
unknown_group_data_waldron['BMI_geq_35'] = unknown_group_data_waldron['BMI'].apply(lambda x: 1 if x >= 35 else 0)
# Grab IDs
group1 = unknown_group_data_waldron[unknown_group_data_waldron.BMI_geq_35 == 0].ID
group2a = unknown_group_data_waldron[unknown_group_data_waldron.BMI_geq_35 == 1].ID
# Annotate
HC_MMC_UMMC_filtered.loc[(HC_MMC_UMMC_filtered.ID.isin(group1)), 'PatientGroup'] = '1'
HC_MMC_UMMC_filtered.loc[(HC_MMC_UMMC_filtered.ID.isin(group2a)), 'PatientGroup'] = '2a'

In [45]:
# These results look much better!
HC_MMC_UMMC_filtered.PatientGroup.isna().sum()

27

## Okay we've done our best to clean metadata, let's drop NaNs
Let's also drop accessory/extra columns that could be leading to duplicate IDs

In [46]:
HC_MMC_UMMC_filtered.aHEI.notna().sum()

1024

In [47]:
HC_MMC_UMMC_filtered.drop(columns=['Status', 'diabetic_status', 'DASH','DDS'], inplace=True)

In [48]:
HC_MMC_UMMC_filtered = HC_MMC_UMMC_filtered.drop_duplicates()

In [49]:
HC_MMC_UMMC_filtered.ID.nunique()

1462

In [50]:
HC_MMC_UMMC_filtered = HC_MMC_UMMC_filtered.dropna(subset=['BMI','Gender','PatientGroup'])

In [51]:
# HC_MMC_UMMC_filtered.SMOKE.notna().sum()

In [52]:
HC_MMC_UMMC_filtered

,ID,PatientGroup,SMOKE,pa_work_2cl,aHEI,activity_h_week,CENTER,AGE,BMI,Gender
0,x10MCx1114,3,0.0,1.0,NaN,NaN,France,59.0,38.154392,Female
1,x10MCx1132,3,0.0,1.0,NaN,NaN,France,73.0,44.634475,Female
3,x10MCx1134,2a,0.0,1.0,18.144745,NaN,France,42.0,44.365888,Female
4,x10MCx1135,3,0.0,1.0,NaN,0.00000,France,57.0,27.719670,Male
6,x10MCx1138,3,1.0,1.0,19.605678,NaN,France,42.0,38.221055,Male
...,...,...,...,...,...,...,...,...,...,...
2824,x30MCx3369,1,NaN,NaN,NaN,NaN,Denmark,54.0,31.057260,Male
2828,x30MCx3375,8,NaN,NaN,NaN,8.90625,Denmark,53.0,19.190550,Female
2829,x30MCx3376,1,NaN,NaN,NaN,NaN,Denmark,68.0,25.016880,Male
2832,x30MCx3379,1,NaN,NaN,NaN,8.84375,Denmark,75.0,23.561940,Female


## Merge with antibiotic use data

In [53]:
# Merge client/covariates with abx use data
abx_count.rename(columns={'SampleID': 'ID'}, inplace=True)
abx_count.drop(columns='PatientGroup', inplace=True)

clients_abxuse = (abx_count.merge(
    HC_MMC_UMMC_filtered, 
    how='right', 
    on='ID').dropna(
    subset='ANTIBIOTICS_TOTAL'))

clients_abxuse

,ID,ANTIBIOTICS_TOTAL,BETA_LACTAM,BISMUTH,FLUOROQUINOLONE,FOSFOMYCIN,FUSIDIC_ACID,GLYCOPEPTIDE,IMIDAZOLE,INHIBITOR,...,UNKNOWN_ANTIBIOTIC,PatientGroup,SMOKE,pa_work_2cl,aHEI,activity_h_week,CENTER,AGE,BMI,Gender
2,x10MCx1134,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2a,0.0,1.0,18.144745,NaN,France,42.0,44.365888,Female
3,x10MCx1135,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3,0.0,1.0,NaN,0.00000,France,57.0,27.719670,Male
4,x10MCx1138,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,3,1.0,1.0,19.605678,NaN,France,42.0,38.221055,Male
6,x10MCx1143,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3,0.0,1.0,26.005617,NaN,France,71.0,39.817729,Female
7,x10MCx1145,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,3,0.0,1.0,41.537608,NaN,France,52.0,31.449660,Female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428,x30MCx3369,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1,NaN,NaN,NaN,NaN,Denmark,54.0,31.057260,Male
1429,x30MCx3375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,8,NaN,NaN,NaN,8.90625,Denmark,53.0,19.190550,Female
1430,x30MCx3376,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1,NaN,NaN,NaN,NaN,Denmark,68.0,25.016880,Male
1431,x30MCx3379,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1,NaN,NaN,NaN,8.84375,Denmark,75.0,23.561940,Female


### Merge with antidiabetic and antilipid medication use data

In [54]:
anti_diab_lipid = anti_diab[['SampleID', 'ANTIDIAB_C', 'ANTILIPID_C']].copy()
anti_diab_lipid.rename(columns={'SampleID': 'ID'}, inplace=True)
abx_med_use = clients_abxuse.merge(anti_diab_lipid, how='left', on=['ID'])
abx_med_use.sort_values(by='ID')

,ID,ANTIBIOTICS_TOTAL,BETA_LACTAM,BISMUTH,FLUOROQUINOLONE,FOSFOMYCIN,FUSIDIC_ACID,GLYCOPEPTIDE,IMIDAZOLE,INHIBITOR,...,SMOKE,pa_work_2cl,aHEI,activity_h_week,CENTER,AGE,BMI,Gender,ANTIDIAB_C,ANTILIPID_C
0,x10MCx1134,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,18.144745,NaN,France,42.0,44.365888,Female,0,0
1,x10MCx1135,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.00000,France,57.0,27.719670,Male,1,0
2,x10MCx1138,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,19.605678,NaN,France,42.0,38.221055,Male,1,0
3,x10MCx1143,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,26.005617,NaN,France,71.0,39.817729,Female,1,0
4,x10MCx1145,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,41.537608,NaN,France,52.0,31.449660,Female,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1287,x30MCx3369,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,Denmark,54.0,31.057260,Male,0,1
1288,x30MCx3375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,8.90625,Denmark,53.0,19.190550,Female,0,0
1289,x30MCx3376,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,Denmark,68.0,25.016880,Male,0,0
1290,x30MCx3379,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,8.84375,Denmark,75.0,23.561940,Female,0,1


## Merge with microbial load

In [55]:
microbial_load = microbes[['ID', 'Microbial load']].copy().drop_duplicates()
microbial_load.rename(columns={'Microbial load': 'microbial_load'}, inplace=True)
abxuse_load = abx_med_use.merge(microbial_load, how='left', on=['ID'])
abxuse_load.sort_values(by='ID')

,ID,ANTIBIOTICS_TOTAL,BETA_LACTAM,BISMUTH,FLUOROQUINOLONE,FOSFOMYCIN,FUSIDIC_ACID,GLYCOPEPTIDE,IMIDAZOLE,INHIBITOR,...,pa_work_2cl,aHEI,activity_h_week,CENTER,AGE,BMI,Gender,ANTIDIAB_C,ANTILIPID_C,microbial_load
0,x10MCx1134,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,18.144745,NaN,France,42.0,44.365888,Female,0,0,NaN
1,x10MCx1135,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,NaN,0.00000,France,57.0,27.719670,Male,1,0,1.432749e+11
2,x10MCx1138,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,19.605678,NaN,France,42.0,38.221055,Male,1,0,NaN
3,x10MCx1143,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,26.005617,NaN,France,71.0,39.817729,Female,1,0,NaN
4,x10MCx1145,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,41.537608,NaN,France,52.0,31.449660,Female,1,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1287,x30MCx3369,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,Denmark,54.0,31.057260,Male,0,1,1.080110e+11
1288,x30MCx3375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,8.90625,Denmark,53.0,19.190550,Female,0,0,1.730000e+11
1289,x30MCx3376,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,Denmark,68.0,25.016880,Male,0,0,1.955090e+11
1290,x30MCx3379,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,8.84375,Denmark,75.0,23.561940,Female,0,1,8.810680e+10


In [56]:
abxuse_load.microbial_load.notna().sum()

595

## Merge with metabolite information prior to regression

In [57]:
# Drop a duplicate col before merging with metabolites
metabolites.drop(columns='Status',inplace=True)

In [58]:
metabolites.set_index('ID', inplace=True)

In [59]:
metabs_special_char = metabolites.columns

In [60]:
metabolites.columns = (
    metabolites.columns
        .str.replace(r'\*', 'star', regex=True)
        .str.replace(r'\+', 'plus', regex=True)
        .str.replace(r"[.\- :,/;']", "_", regex=True)
        .str.replace(r"[\(\)\[\]]", "", regex=True)
        .str.replace(r"_+", "_", regex=True)   # collapse ___ to _
        .str.strip("_")                        # remove leading/trailing _
)

In [61]:
metabs_all_clean = metabolites.columns

In [62]:
# Their data was log-transformed already
# Standardize to match Arivale workflow
metabolites = metabolites.apply(lambda x: (x - x.mean()) / x.std())

In [63]:
metabolites.reset_index(inplace=True)

In [64]:
# Merge metabolites with clients and abx use
clients_abx_metabs = abxuse_load.merge(metabolites.drop_duplicates(),
                                       how='left', on='ID')

In [65]:
metabolites.columns[metabolites.columns.str.contains("dodecadienoate")]

Index(['dodecadienoate_12_2star'], dtype='object')

In [66]:
clients_abx_metabs.rename(columns=dict(
    zip(metabs_all_clean, "metabolite_" + metabs_all_clean)), inplace=True)

In [67]:
arivale_sig_clean = [
    '2_hydroxy_3_methylvalerate',
    '2_hydroxybutyrate_2_hydroxyisobutyrate',
    '2_piperidinone',
    '3_3_hydroxyphenylpropionate',
    '3_4_hydroxyphenyllactate_HPLA',
    '3_hydroxy_2_ethylpropionate',
    '3_hydroxydecanoate', 
    '3_hydroxylaurate',
    '3_hydroxymyristate',
    '3_hydroxyoctanoate',
    '3_phenylpropionate_hydrocinnamate',
    '4_hydroxyhippurate',
    '4_methyl_2_oxopentanoate',
    '5_hydroxyhexanoate',
    '5alpha_androstan_3beta_17alpha_diol_disulfate',
    'X_12216',
    'X_12544',
    'X_15728',
    'X_17335',
    'X_21353', # Removed extra 2 underscores from unknowns
    'acetylcarnitine_C2',
    'alpha_hydroxyisocaproate',
    'alpha_hydroxyisovalerate',
    'alpha_ketobutyrate',
    'cinnamoylglycine',
    'dodecadienoate_12_2star',
    'glutarate_C5_DC',
    'hippurate',
    'hydantoin_5_propionate',
    'indolepropionate',
    'isoursodeoxycholate',
    'phenyllactate_PLA',
]

In [68]:
# Add metabolite_ prefix to make names iterable in regressions
arivale_sig_clean = ["metabolite_" + item for item in arivale_sig_clean]

## Regression
Includes health status and nationality as a covariate.

In [69]:
len(set(clients_abx_metabs.columns).intersection(set(arivale_sig_clean)))

32

In [70]:
clients_abx_metabs['abx_binary'] = clients_abx_metabs['ANTIBIOTICS_TOTAL'].apply(lambda x: 1 if x >= 1 else 0)

In [71]:
clients_abx_metabs.columns[20:50]

Index(['TETRACYCLINE', 'TRIMETHOPRIM', 'UNKNOWN_ANTIBIOTIC', 'PatientGroup',
       'SMOKE', 'pa_work_2cl', 'aHEI', 'activity_h_week', 'CENTER', 'AGE',
       'BMI', 'Gender', 'ANTIDIAB_C', 'ANTILIPID_C', 'microbial_load',
       'metabolite_acetate', 'metabolite_acetone', 'metabolite_artemisin',
       'metabolite_beta_sitosterol', 'metabolite_betaine',
       'metabolite_betaine_aldehyde', 'metabolite_butyrylcarnitine',
       'metabolite_catechol', 'metabolite_cellotetraose', 'metabolite_choline',
       'metabolite_D_trehalose', 'metabolite_D_lyxose', 'metabolite_D_malate',
       'metabolite_D_sorbitol', 'metabolite_D_threitol'],
      dtype='object')

In [82]:
forumla_metacardis_total = ('{feature} ~ C(Gender) + AGE + BMI + microbial_load + C(PatientGroup) + C(CENTER) + '
                           'C(ANTILIPID_C) + C(ANTIDIAB_C) + ANTIBIOTICS_TOTAL')
forumla_metacardis_binary = ('{feature} ~ C(Gender) + AGE + BMI + microbial_load + C(PatientGroup) + C(CENTER) +'
                           'C(ANTILIPID_C) + C(ANTIDIAB_C) + C(abx_binary)')

In [83]:
tests_metacardis_mets_total = run_tests(arivale_sig_clean, clients_abx_metabs, forumla_metacardis_total, "ANTIBIOTICS_TOTAL")
tests_metacardis_mets_bin = run_tests(arivale_sig_clean, clients_abx_metabs, forumla_metacardis_binary, "C(abx_binary)[T.1]")

Output()

Output()

In [84]:
tests_metacardis_mets_total['feature'] = tests_metacardis_mets_total['feature'].str.replace('metabolite_', '', regex=False)
tests_metacardis_mets_bin['feature'] = tests_metacardis_mets_bin['feature'].str.replace('metabolite_', '', regex=False)

In [85]:
tests_metacardis_mets_sig_tot = tests_metacardis_mets_total[tests_metacardis_mets_total['q'] < 0.1].sort_values(by='q')
tests_metacardis_mets_sig_tot

,feature,beta,se,t_statistic,conf_low,conf_high,p,n,q
metabolite_X_12544,X_12544,-0.100819,0.030588,-3.296033,-0.160897,-0.040742,0.001041,590.0,0.033306
metabolite_cinnamoylglycine,cinnamoylglycine,-0.080943,0.029252,-2.767098,-0.138396,-0.023490,0.005837,590.0,0.093395


In [86]:
tests_metacardis_mets_sig_bin = tests_metacardis_mets_bin[tests_metacardis_mets_bin['q'] < 0.1].sort_values(by='q')
tests_metacardis_mets_sig_bin

,feature,beta,se,t_statistic,conf_low,conf_high,p,n,q
metabolite_X_12544,X_12544,-0.277911,0.077216,-3.599141,-0.429570,-0.126253,0.000347,590.0,0.011095
metabolite_4_hydroxyhippurate,4_hydroxyhippurate,0.248130,0.081033,3.062096,0.088975,0.407284,0.002300,590.0,0.036805


Create two different columns to observe this pattern 

In [87]:
tests_metacardis_mets_total.rename(columns={
    'beta':'beta_total',
    'q':'q_total',
    'feature':'CHEMICAL_NAME'
}, inplace=True)

tests_metacardis_mets_bin.rename(columns={
    'beta':'beta_binary',
    'q':'q_binary',
    'feature':'CHEMICAL_NAME'
}, inplace=True)

In [88]:
tests_metacardis_mets_bin.to_csv("data/metacardis_regression_tests_binary_abx.csv", index=False)
tests_metacardis_mets_total.to_csv("data/metacardis_regression_tests_total_abx.csv", index=False)

Use the small df below to merge onto Arivale data tables

In [89]:
metacardis_results = tests_metacardis_mets_total[['CHEMICAL_NAME','beta_total','q_total',]].copy().merge(
    tests_metacardis_mets_bin[['CHEMICAL_NAME','beta_binary','q_binary']].copy(),
    how='left',
    on='CHEMICAL_NAME'
)

In [90]:
metacardis_results.CHEMICAL_NAME = metacardis_results.CHEMICAL_NAME.str.replace(
    '3_4_hydroxyphenyllactate_HPLA','3_4_hydroxyphenyllactate')

# Merge onto Arivale data

In [91]:
metab_betas = pd.read_csv("data/rh_metabs_betas.csv")
metab_betas.set_index('CHEMICAL_NAME', inplace=True)

metab_qs = pd.read_csv("data/rh_metabs_qs.csv")
metab_qs.set_index('CHEMICAL_NAME', inplace=True)

anns_snapshot = pd.read_csv("data/metabolites_snapshot_updated.csv")
anns = anns_snapshot[['CHEMICAL_NAME','SUB_PATHWAY','SUPER_PATHWAY']].copy()
anns.set_index('CHEMICAL_NAME', inplace=True)

metab_betas = metab_betas.merge(anns, how='left', on='CHEMICAL_NAME')

In [92]:
len(metab_betas)

34

I'll have to harmonize the chemical names for merging with the MetaCardis data. Let's create a new var to hold the plot names. 

In [93]:
plot_labs = metab_betas.index.tolist()
metab_betas['plot_name'] = plot_labs
metab_qs['plot_name'] = plot_labs

In [94]:
metab_betas.reset_index(inplace=True)
metab_qs.reset_index(inplace=True)

In [95]:
# Replace special characters
metab_betas.CHEMICAL_NAME = (
    metab_betas.CHEMICAL_NAME
        .str.replace(r'\*', 'star', regex=True)
        .str.replace(r'\+', 'plus', regex=True)
        .str.replace(r"[.\- :,/;']", "_", regex=True)
        .str.replace(r"[\(\)\[\]]", "", regex=True)
        .str.replace(r"_+", "_", regex=True)   # collapse ___ to _
        .str.strip("_")                        # remove leading/trailing _
)

metab_qs.CHEMICAL_NAME = (
    metab_qs.CHEMICAL_NAME
        .str.replace(r'\*', 'star', regex=True)
        .str.replace(r'\+', 'plus', regex=True)
        .str.replace(r"[.\- :,/;']", "_", regex=True)
        .str.replace(r"[\(\)\[\]]", "", regex=True)
        .str.replace(r"_+", "_", regex=True)   # collapse ___ to _
        .str.strip("_")                        # remove leading/trailing _
)

In [96]:
clean_chem_names = metab_betas.CHEMICAL_NAME.tolist()
clean_chem_names

['3_4_hydroxyphenyllactate',
 'glutarate_C5_DC',
 'alpha_ketobutyrate',
 '3_phenylpropionate_hydrocinnamate',
 'hippurate',
 '4_methyl_2_oxopentanoate',
 'alpha_hydroxyisocaproate',
 '3_hydroxyoctanoate',
 'phenyllactate_PLA',
 'acetylcarnitine_C2',
 '3_hydroxymyristate',
 '3_hydroxydecanoate',
 'indolepropionate',
 '3_hydroxylaurate',
 '5_hydroxyhexanoate',
 '3_hydroxy_2_ethylpropionate',
 'alpha_hydroxyisovalerate',
 '4_hydroxyhippurate',
 '2_hydroxy_3_methylvalerate',
 '3_3_hydroxyphenylpropionate',
 '5alpha_androstan_3beta_17alpha_diol_disulfate',
 'cinnamoylglycine',
 'isoursodeoxycholate',
 'hydantoin_5_propionate',
 '2_piperidinone',
 '2_hydroxybutyrate_2_hydroxyisobutyrate',
 'dodecadienoate_12_2star',
 'X_21353',
 'X_12544',
 'X_12216',
 'X_15728',
 'X_12283',
 'X_17335',
 'X_17351']

In [117]:
metab_betas = metab_betas.merge(
    metacardis_results[['CHEMICAL_NAME','beta_total','beta_binary']].copy(),
    how='left',
    on='CHEMICAL_NAME')

In [118]:
metab_qs = metab_qs.merge(
    metacardis_results[['CHEMICAL_NAME','q_total','q_binary']].copy(),
    how='left',
    on='CHEMICAL_NAME')

In [119]:
metab_betas

,CHEMICAL_NAME,abx_three,abx_twelve,chol_meds,blood_sugar_meds,SUB_PATHWAY,SUPER_PATHWAY,plot_name,beta_total,beta_binary
0,glutarate_C5_DC,0.386387,0.098197,0.051383,0.285104,"Fatty Acid, Dicarboxylate",Lipid,glutarate (C5-DC),-0.003295,-0.063899
1,alpha_ketobutyrate,-0.290272,-0.022889,0.033312,0.609669,"Methionine, Cysteine, SAM and Taurine Metabolism",Amino Acid,alpha-ketobutyrate,-0.002719,-0.108508
2,3_phenylpropionate_hydrocinnamate,-0.425302,-0.207168,-0.076640,-0.031335,Benzoate Metabolism,Xenobiotics,3-phenylpropionate (hydrocinnamate),-0.053912,-0.126921
3,hippurate,-0.310749,-0.213062,-0.165728,-0.151796,Benzoate Metabolism,Xenobiotics,hippurate,-0.011818,0.032898
4,4_methyl_2_oxopentanoate,-0.235198,-0.099729,0.020687,0.532149,"Leucine, Isoleucine and Valine Metabolism",Amino Acid,4-methyl-2-oxopentanoate,0.004964,0.043781
5,alpha_hydroxyisocaproate,-0.240082,-0.094191,-0.047424,0.281556,"Leucine, Isoleucine and Valine Metabolism",Amino Acid,alpha-hydroxyisocaproate,0.033590,0.089837
6,phenyllactate_PLA,-0.231256,-0.120232,-0.058598,0.115497,Phenylalanine Metabolism,Amino Acid,phenyllactate (PLA),0.025727,0.056199
7,acetylcarnitine_C2,-0.319702,-0.066922,-0.123283,0.115206,"Fatty Acid Metabolism (Acyl Carnitine, Short C...",Lipid,acetylcarnitine (C2),0.006265,0.017558
8,indolepropionate,-0.257387,-0.316069,-0.046714,-0.327767,Tryptophan Metabolism,Amino Acid,indolepropionate,-0.004826,-0.039206
9,3_hydroxylaurate,-0.265493,-0.135663,-0.076200,0.318170,"Fatty Acid, Monohydroxy",Lipid,3-hydroxylaurate,-0.014779,-0.079906


# Read in data from SOL

In [97]:
sol_results_three = pd.read_excel("data/SOL_metab_res_no_imp.xlsx", sheet_name=0, header=0)
sol_results_six = pd.read_excel("data/SOL_metab_res_no_imp.xlsx", sheet_name=1, header=0)

In [98]:
sol_results_three.chemical_name = sol_results_three.chemical_name.str.replace(
    'N-lactoyl leucine (1-carboxyethyl leucine)','N-lactoyl leucine')

sol_results_six.chemical_name = sol_results_six.chemical_name.str.replace(
    'N-lactoyl leucine (1-carboxyethyl leucine)','N-lactoyl leucine')

In [99]:
sol_results_three.rename(columns={
    'chemical_name':'plot_name', 
    'est':'sol_beta_three',
    'q':'sol_q_three'
}, inplace=True)

sol_results_six.rename(columns={
    'chemical_name':'plot_name', 
    'est':'sol_beta_six',
    'q':'sol_q_six'
}, inplace=True)

In [100]:
metab_betas = metab_betas.merge(
    sol_results_three[['plot_name','sol_beta_three']].copy(),
    how='left',
    on='plot_name')

metab_betas = metab_betas.merge(
    sol_results_six[['plot_name','sol_beta_six']].copy(),
    how='left',
    on='plot_name')

In [101]:
metab_qs = metab_qs.merge(
    sol_results_three[['plot_name','sol_q_three']].copy(),
    how='left',
    on='plot_name')

metab_qs = metab_qs.merge(
    sol_results_six[['plot_name','sol_q_six']].copy(),
    how='left',
    on='plot_name')

## Diener annotations

In [102]:
diener_interactions = pd.read_excel("data/diener_2022_gene_microbe_interactions.xlsx", sheet_name=2, header=0)

In [103]:
diener_interactions_label = diener_interactions[['BIOCHEMICAL_NAME','n_taxa','micro_r2', 'geno_r2', 'total', 'group']].copy()
diener_interactions_label.rename(columns={'BIOCHEMICAL_NAME':'plot_name'}, inplace=True)

In [104]:
diener_interactions_label.plot_name = diener_interactions_label.plot_name.str.replace(
    '1-carboxyethylleucine', 'N-lactoyl leucine')

In [105]:
diener_interactions_label.plot_name[diener_interactions_label.plot_name.str.contains("propionate")]

14                   3-phenylpropionate (hydrocinnamate)
68                                      indolepropionate
154                                 imidazole propionate
296                          3-hydroxy-2-ethylpropionate
325    3-carboxy-4-methyl-5-pentyl-2-furanpropionate ...
415                                   3-ureidopropionate
547                               hydantoin-5-propionate
Name: plot_name, dtype: object

In [106]:
metab_betas = metab_betas.merge(
    diener_interactions_label, 
    on='plot_name',
    how='left')

In [75]:
# metab_betas

In [107]:
metab_betas.fillna({"SUPER_PATHWAY": "Unknown"}, inplace=True)
metab_betas.fillna({"SUB_PATHWAY": "Unknown"}, inplace=True)
metab_betas.fillna({"group": "N/A"}, inplace=True)

In [108]:
metab_betas.group = metab_betas.group.str.replace("hybrid","Hybrid")
metab_betas.group = metab_betas.group.str.replace("microbiome","Microbiome")
metab_betas.group = metab_betas.group.str.replace("genetics","Genetics")

In [109]:
metab_betas.rename(columns={'group':'Determinant_Association'}, inplace=True)

In [138]:
# metab_betas

In [110]:
metab_betas.to_csv('data/metabolite_abx_regressions_betas.csv', index=False)
metab_qs.to_csv('data/metabolite_abx_regressions_qs.csv', index=False)